# 概要
[きのこを分類](https://www.kaggle.com/datasets/uciml/mushroom-classification/data)するAIを作成します。

# 対象ファイル
mushrooms.csv

In [128]:
import pandas as pd

original_data = pd.read_csv('./mushrooms.csv')

print(original_data.head())

  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                      w         p          w   
1       

In [129]:
# データの全体を確認します。
print(original_data.info())
print(original_data.describe())
print(original_data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   class                     8124 non-null   object
 1   cap-shape                 8124 non-null   object
 2   cap-surface               8124 non-null   object
 3   cap-color                 8124 non-null   object
 4   bruises                   8124 non-null   object
 5   odor                      8124 non-null   object
 6   gill-attachment           8124 non-null   object
 7   gill-spacing              8124 non-null   object
 8   gill-size                 8124 non-null   object
 9   gill-color                8124 non-null   object
 10  stalk-shape               8124 non-null   object
 11  stalk-root                8124 non-null   object
 12  stalk-surface-above-ring  8124 non-null   object
 13  stalk-surface-below-ring  8124 non-null   object
 14  stalk-color-above-ring  

今回は一列目のclassを最終の分類の対象にして、他のデータからclassを判断するようにします。

初期データが22列を持ちまして、データの量が多い、処理するに時間がかかります。

お互いの関係があるかもしれませんので、まずは前処理で主成分分析で次元削減してから、データを学習させます。

In [130]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import Binarizer

# ラベルエンコードと標準化を行います。
X = pd.DataFrame()
label_encoder = LabelEncoder()

Y = label_encoder.fit_transform(original_data['class'])
input_columns = pd.get_dummies(data=original_data, drop_first=True)
# binarizer = Binarizer(threshold=0.5)
for column in input_columns.columns:
    X[column] = label_encoder.fit_transform(input_columns[column])

print(X.shape, Y.shape)
print(X[:5])
print(Y[:5])

(8124, 96) (8124,)
   class_p  cap-shape_c  cap-shape_f  cap-shape_k  cap-shape_s  cap-shape_x  \
0        1            0            0            0            0            1   
1        0            0            0            0            0            1   
2        0            0            0            0            0            0   
3        1            0            0            0            0            1   
4        0            0            0            0            0            1   

   cap-surface_g  cap-surface_s  cap-surface_y  cap-color_c  ...  \
0              0              1              0            0  ...   
1              0              1              0            0  ...   
2              0              1              0            0  ...   
3              0              0              1            0  ...   
4              0              1              0            0  ...   

   population_n  population_s  population_v  population_y  habitat_g  \
0             0          

In [131]:
from sklearn.decomposition import PCA

pca = PCA()
pca.fit_transform(X)
# print(pca.explained_variance_)
# print(pca.explained_variance_ratio_)
explained_variance_ratio_list = pca.explained_variance_ratio_.cumsum()
print(explained_variance_ratio_list)

for i, ratio in enumerate(explained_variance_ratio_list):
    if ratio > 0.98:
        print(i + 1)
        break
print(f'{i + 1}次元で{(ratio * 100):.2f}以上の情報を保持できます。')
dimensions = i + 1

[0.18780516 0.29946978 0.38858929 0.44058912 0.483378   0.52507418
 0.56096367 0.59221469 0.61756826 0.64230743 0.66401481 0.6855858
 0.70471571 0.72232022 0.73977002 0.75660992 0.77240484 0.78711566
 0.8007198  0.81339716 0.82533689 0.83629224 0.84668488 0.8568008
 0.86666197 0.87601647 0.88477221 0.89338498 0.9016464  0.90925867
 0.91627325 0.92279813 0.9289885  0.93447664 0.93974086 0.94486243
 0.94943885 0.95374836 0.95785897 0.96184845 0.96555326 0.96890791
 0.9718283  0.97450805 0.97707467 0.97930656 0.98111611 0.98274465
 0.98418663 0.98542016 0.98658279 0.98772869 0.98873985 0.98972459
 0.99064861 0.99155088 0.99236641 0.99308832 0.99376858 0.99440294
 0.99500815 0.99550868 0.9959642  0.99638728 0.99678714 0.99716975
 0.99754435 0.9978579  0.99814792 0.99840967 0.99862467 0.99883612
 0.999039   0.99923219 0.99940476 0.99953194 0.99963135 0.99971725
 0.99978812 0.99984477 0.99989583 0.99993929 0.99997556 0.99999616
 1.         1.         1.         1.         1.         1.
 1.  

上記の結果を見たら、元の22列のXデータは実際19列くらいで 97.9% の状況を分析できます。それら主成分分析できます。
残りのデータは誤差か分析しづらい状況と認識して、データから除外します。

In [132]:
pca_main = PCA(n_components=dimensions)
X = pca_main.fit_transform(X)
print(X.shape)

(8124, 47)


In [133]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.3, random_state=42)
print(X_train.shape, X_valid.shape, X_test.shape, Y_train.shape, Y_valid.shape, Y_test.shape)

(3980, 47) (1706, 47) (2438, 47) (3980,) (1706,) (2438,)


In [135]:
# Optunaを使ってハイパーパラメータの最適化を行います。
from catboost import CatBoostClassifier
from optuna_integration import CatBoostPruningCallback
import optuna

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 2, 2000),
        "depth": trial.suggest_int("depth", 2, 16),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.8, log=True),
        "l2_leaf_reg":   trial.suggest_float("l2_leaf_reg", 1e-2, 100.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-8, 10.0, log=True),
        # "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "rsm": trial.suggest_float("rsm", 0.5, 1.0),
        "border_count": trial.suggest_int("border_count", 64, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),
        "od_type": "Iter",
        "eval_metric": "AUC",
        "od_wait": trial.suggest_int("od_wait", 20, 100),
        "random_seed": 42,
        "verbose": False,
        "loss_function": "Logloss"
    }

    params["bootstrap_type"] = trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])
    if params["bootstrap_type"] == "Bernoulli":
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
    if params["grow_policy"] in ("Depthwise", "Lossguide"):
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 1, 64)

    model = CatBoostClassifier(**params)

    # Optunaのプルーニング用コールバック（検証AUCを監視）
    pruning_cb = CatBoostPruningCallback(trial, metric="AUC")

    model.fit(
        X_train, Y_train,
        eval_set=(X_valid, Y_valid),
        use_best_model=True,
        # CatBoost は user callback を受け取れます（v0.26+）
        callbacks=[pruning_cb],
    )

    pruning_cb.check_pruned()

    return model.get_best_score()["validation"]["AUC"]

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=50),
)
study.optimize(objective, n_trials=500, show_progress_bar=True)

print("Best value:", study.best_value)
print("Best params:", study.best_trial.params)


[I 2025-09-19 15:17:28,592] A new study created in memory with name: no-name-5f678c33-81d5-4670-aa4d-be95ac36d5de


  0%|          | 0/500 [00:00<?, ?it/s]

/var/folders/g4/4w4r7d157h93395l0v75s7dc0000gn/T/ipykernel_93297/3841582269.py:34: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_cb = CatBoostPruningCallback(trial, metric="AUC")


[I 2025-09-19 15:17:28,834] Trial 0 finished with value: 1.0 and parameters: {'iterations': 1199, 'depth': 9, 'learning_rate': 0.00033286375898150446, 'l2_leaf_reg': 0.027571428766436208, 'random_strength': 5.0981376717050116e-08, 'rsm': 0.801953844692473, 'border_count': 252, 'grow_policy': 'Lossguide', 'od_wait': 50, 'bootstrap_type': 'Bayesian', 'min_data_in_leaf': 34}. Best is trial 0 with value: 1.0.
[I 2025-09-19 15:17:29,242] Trial 1 finished with value: 1.0 and parameters: {'iterations': 1837, 'depth': 14, 'learning_rate': 0.037179380713290675, 'l2_leaf_reg': 0.04752623593722191, 'random_strength': 4.307392770396258e-06, 'rsm': 0.5296892681001512, 'border_count': 249, 'grow_policy': 'Depthwise', 'od_wait': 81, 'bootstrap_type': 'Bayesian', 'min_data_in_leaf': 60}. Best is trial 0 with value: 1.0.
[I 2025-09-19 15:17:29,521] Trial 2 finished with value: 1.0 and parameters: {'iterations': 1992, 'depth': 2, 'learning_rate': 0.03499365188128647, 'l2_leaf_reg': 0.16297039895535195, 

In [137]:
print(study)
print(study.best_trial)

FrozenTrial(number=0, state=1, values=[1.0], datetime_start=datetime.datetime(2025, 9, 19, 15, 17, 28, 600039), datetime_complete=datetime.datetime(2025, 9, 19, 15, 17, 28, 834194), params={'iterations': 1199, 'depth': 9, 'learning_rate': 0.00033286375898150446, 'l2_leaf_reg': 0.027571428766436208, 'random_strength': 5.0981376717050116e-08, 'rsm': 0.801953844692473, 'border_count': 252, 'grow_policy': 'Lossguide', 'od_wait': 50, 'bootstrap_type': 'Bayesian', 'min_data_in_leaf': 34}, user_attrs={}, system_attrs={}, intermediate_values={0: 0.99764520432679, 1: 0.9994464188837591, 2: 0.9999321863132605, 3: 0.9999972320944188, 4: 0.9999986160472094, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 0.9999986160472094, 11: 0.9999986160472094, 12: 0.9999986160472094, 13: 0.9999986160472094, 14: 0.9999986160472094, 15: 0.9999986160472094, 16: 0.9999986160472094, 17: 0.9999986160472094, 18: 0.9999986160472094, 19: 0.9999972320944188, 20: 0.9999944641888376, 21: 0.9999944641888376, 22: 0.999995848141

In [138]:
# 最良パラメータで最終学習
best_params = {
    **study.best_trial.params,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "task_type": "CPU",
    "verbose": 100,
}
print(best_params)
final_model = CatBoostClassifier(**best_params)
final_model.fit(X_train, Y_train, eval_set=(X_valid, Y_valid), use_best_model=True)

{'iterations': 1199, 'depth': 9, 'learning_rate': 0.00033286375898150446, 'l2_leaf_reg': 0.027571428766436208, 'random_strength': 5.0981376717050116e-08, 'rsm': 0.801953844692473, 'border_count': 252, 'grow_policy': 'Lossguide', 'od_wait': 50, 'bootstrap_type': 'Bayesian', 'min_data_in_leaf': 34, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'random_seed': 42, 'task_type': 'CPU', 'verbose': 100}
0:	test: 0.9976452	best: 0.9976452 (0)	total: 3.23ms	remaining: 3.87s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1
bestIteration = 5

Shrink model to first 6 iterations.


In [139]:
Y_pred = final_model.predict(X_test)

print(Y_pred[:10])

[0 1 1 0 1 1 1 1 0 0]


In [140]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1257
           1       1.00      0.99      1.00      1181

    accuracy                           1.00      2438
   macro avg       1.00      1.00      1.00      2438
weighted avg       1.00      1.00      1.00      2438



In [ ]:
final_model.save_model("catboost_model.cbm")